In [55]:
from manim import *
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
%%manim -ql -v WARNING BannerExample

config.media_width = "75%"
config.media_embed = True
Mobject.set_default(color=WHITE)
class BannerExample(MovingCameraScene):
    def construct(self):
        max_quotient = 4
        modulo = 12
        areas_cmap = plt.get_cmap('Blues')
        
        circle = Circle(radius=3.5)
        circle.set_stroke()
        point_start = circle.get_start()
        self.add(circle)

        hours_coords = [circle.point_at_angle(PI / 2 - mod * 2 * PI / modulo) for mod in range(modulo)]
        hours_texts = []
        for mod in range(modulo):
            t = Text(f"{modulo * int(mod==0) + mod}", font_size=36).move_to(hours_coords[mod])
            towards_center = ORIGIN - t.get_center()
            t.shift(0.2*towards_center)
            hours_texts.append(t)

        arm = Line(ORIGIN, 2*UP)
        self.play(*[Write(h) for h in hours_texts])
        self.play(Create(arm))
        self.camera.frame.save_state()
        classes_objs = {m: [] for m in range(modulo)}
        classes_areas = []
        for quotient in range(max_quotient-1):
            for mod in range(modulo):
                ht = hours_texts[mod]
                towards_center = ORIGIN - hours_coords[mod]
                int_coords = -0.3 * (quotient+1) * towards_center + ht.get_center()
                int_txt = Text(f"{mod + quotient*modulo}", font_size=24).move_to(int_coords)
                classes_objs[mod].append(int_txt)

                if quotient == max_quotient - 2:
                    color = ManimColor.from_rgba(areas_cmap(6 / modulo)).opacity(0.3)
                    area = AnnularSector(
                            0, 1.1*np.sqrt(((int_coords - ORIGIN)**2).sum()),
                            -2 * PI / modulo,
                            PI / 2 + PI / modulo - mod * 2 * PI / modulo,
                            stroke_width=2,
                            stroke_color=BLACK,
                            fill_opacity=0.3,
                            fill_color=ManimColor.from_rgba(color),
                            z_index=-1,
                        )#.set_colors_by_gradient(color, WHITE)
                    classes_areas.append(area)

        animations = []
        for quotient in range(max_quotient-1):
            quotient_anims = []
            for mod in range(modulo):
                ht = hours_texts[mod]
                int_txt = classes_objs[mod][quotient]
                quotient_anims.append(GrowFromPoint(int_txt, ht))
                quotient_anims.append(Rotate(arm, -2 * PI / modulo, about_point=ORIGIN))
            # Run times set below are not absolute, but relative to one another. Needed so zoom is not instant.
            quotient_seq = Succession(*quotient_anims, run_time=4)
            zoom_anim = self.camera.auto_zoom([classes_objs[mod][quotient] for mod in classes_objs.keys()], margin=1, animate=True).set_run_time(1)
            animations.append(AnimationGroup(zoom_anim, quotient_seq))

        self.play(Succession(*animations, run_time=10, rate_func=rate_functions.ease_in_sine))
        self.play(*[GrowFromPoint(a, ORIGIN) for a in classes_areas])
        self.play(Wait(1), Restore(self.camera.frame), Uncreate(arm))

        class_texts = []
        for i,t in enumerate(hours_texts):
            class_txt = Text(f"[{t.text}]").move_to(t)
            if i == 0:
                class_txt = VGroup(class_txt.move_to(t), Text("="), Text("[0]")).arrange(DOWN).move_to(t.get_top(), aligned_edge=np.array([0., 1., 0.]))
            class_texts.append(class_txt)
        self.play(AnimationGroup(*[Transform(t, ct) for t, ct in zip(hours_texts, class_texts)]))
        self.play(Wait(1))